# Qwen3-4B text-to-SQL — large fine-tune on Kaggle

Scales up the run that was done locally on an M5 Pro. The laptop version used
4,000 Gretel examples; this uses **20,000 examples across two datasets** on a
free Kaggle T4.

**Sidebar settings (required):**

| Setting | Value |
|---|---|
| Accelerator | `GPU T4 x2` |
| Internet | **On** |

No Hugging Face token is needed. The notebook saves the adapter and the
evaluation report as notebook output; publishing to the Hub happens locally
afterwards, so no secret ever enters this kernel.

Source: https://github.com/garvbahl37-gif/text2sql-qwen3-finetune.git

In [ ]:
# --- 1. Hardware check (stops here if the GPU is unusable) ------------------
import subprocess, sys, torch

name = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
major, minor = torch.cuda.get_device_capability()
print(f"GPU        : {name}")
print(f"capability : {major}.{minor}")
print(f"torch      : {torch.__version__}   bf16: {torch.cuda.is_bf16_supported()}")

if major < 7:
    raise SystemExit(
        f"\nSTOP. This session has a compute-capability {major}.{minor} GPU "
        f"({name.split(',')[0]}).\n"
        "Modern PyTorch and Unsloth builds ship no kernels for it, so training "
        "would fail with\n'CUDA error: no kernel image is available for execution "
        "on the device'.\n\n"
        "FIX: in the right-hand sidebar set Accelerator to 'GPU T4 x2', then "
        "Run All again.\n"
        "The GPU type cannot be set through the Kaggle API, only in this UI."
    )
print("\nGPU is supported." + ("" if torch.cuda.is_bf16_supported() else " Turing has no bf16, so fp16 is selected automatically."))

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub

In [ ]:
# --- 2. Verify the install before spending GPU time on it --------------------
try:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    import trl, peft, transformers
    print(f"ok | transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")
except Exception as e:
    print("INSTALL FAILED:", type(e).__name__, e)
    print("\nFallback, then Run > Restart session and skip the install cell:")
    print("  !pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo")

In [ ]:
# --- 3. Get the code --------------------------------------------------------
import os, shutil, subprocess
from pathlib import Path

WORK = Path("/kaggle/working/ft")
if not (WORK / "train.py").exists():
    shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
    subprocess.run(["git","clone","--depth","1","https://github.com/garvbahl37-gif/text2sql-qwen3-finetune.git","/kaggle/working/repo"], check=True)
    shutil.copytree("/kaggle/working/repo/training", WORK, dirs_exist_ok=True)
os.chdir(WORK)
print("cwd:", os.getcwd())
print("files:", sorted(p.name for p in Path('.').glob('*.py')))

## 4. Build a larger, more diverse training set

Two sources instead of one:

- `gretelai/synthetic_text_to_sql` — synthetic, but ships INSERT rows, so gold
  SQL is verified by running it and the eval can compare result sets.
- `b-mc2/sql-create-context` — derived from Spider and WikiSQL, so real-world
  phrasing and schemas. CREATE TABLE only, so its gold SQL is validated by
  executing against the empty schema, which still proves the syntax parses and
  every table and column reference resolves.

`--balance` allocates the budget by measured headroom rather than the natural
mix, which is 54% "basic SQL" the base model already handles.

In [ ]:
# --- helper: run a step and STOP if it fails --------------------------------
# `!python foo.py` returns a non-zero exit code on failure but Jupyter does not
# raise, so the notebook happily continues and dies later somewhere confusing.
import subprocess, sys

def step(cmd: str):
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise SystemExit(f"\nStep failed with exit code {p.returncode}:\n  {cmd}\n"
                         "Nothing after this will work, so the notebook stops here.")
    print(f"\nok: {cmd.split()[1] if len(cmd.split()) > 1 else cmd}", flush=True)

step("python prepare_data.py --sources gretel,createcontext "
     "--train-size 20000 --val-size 400 --test-size 300 --balance")

## 5. Train

Larger batch than the Apple Silicon run because a T4's 16GB is dedicated,
whereas unified memory is shared with the OS. `--max-seq 640` is measured from
the data: median 194 tokens, longest 581.

**Watch the first 50 steps.** The `it/s` in the progress bar tells you the real
runtime on whichever T4 you were given — multiply it out before walking away.
If it reports out-of-memory, halve `--batch-size` and double `--grad-accum`.

In [ ]:
step("python train.py --data data --out outputs/qwen3-4b-text2sql-lora "
     "--max-seq 640 --batch-size 16 --grad-accum 2 --rank 32 --epochs 1")

In [ ]:
step("python evaluate.py --adapter outputs/qwen3-4b-text2sql-lora --limit 300")

## 6. Save the results

The adapter and `eval_report.json` are written to `/kaggle/working`, which
becomes the notebook's downloadable Output. Locally:

```
kaggle kernels output gb1105/qwen3-text2sql-finetune -p ./kaggle_out
cp kaggle_out/eval_report.json web/data/eval_report.json
python training/fuse_and_push_mlx.py --adapter <downloaded adapter> --repo bharatverse11/qwen3-4b-text2sql
```

In [ ]:
import json, shutil
from pathlib import Path

out = Path("/kaggle/working")
shutil.copy("outputs/eval_report.json", out / "eval_report.json")
adapter = Path("outputs/qwen3-4b-text2sql-lora")
shutil.make_archive(str(out / "adapter"), "zip", adapter)

r = json.loads((out / "eval_report.json").read_text())
b, t = r["metrics"]["base"], r["metrics"]["tuned"]
print(f"{'metric':<22}{'base':>10}{'tuned':>12}{'change':>11}")
print("-" * 55)
for k in b:
    print(f"{k:<22}{b[k]:>9.1%}{t[k]:>11.1%}{(t[k]-b[k])*100:>+10.1f}")
c = r["counts"]
print(f"\nwins {c['tuned_win']} | both right {c['both_correct']} "
      f"| regressions {c['tuned_regression']} | both wrong {c['both_wrong']}")
print("\nby complexity:")
for name, v in r.get("by_complexity", {}).items():
    print(f"  {name:<20} n={v['n']:<5} {v['base']:>6.1%} -> {v['tuned']:>6.1%}")
print("\nSaved to /kaggle/working - download from the Output panel.")